# 🔄 LTGG Dashboard – data refresh

Run the cells below in **Google Colab** to convert a fresh copy of `LTGG_Trades_and_Performance.xlsx` into the `data.pkl` file consumed by the Streamlit dashboard.

The workbook should have three sheets: **Trades**, **Performance**, and **Current Portfolio**.

**Workflow**
1. Run cell 1 – uploads the Excel workbook from your computer.
2. Run cell 2 – builds `data.pkl` (prints a sanity-check summary).
3. Run cell 3 – downloads `data.pkl` to your computer.
4. Replace the `data.pkl` in your GitHub repo and push. Streamlit Cloud redeploys automatically.

> If a holding rebrands (e.g. Beigene → BeOne Medicines), add it to the `NAME_ALIASES` dict at the top of the build cell.

> Optional cell 4 at the bottom pushes the pickle directly to GitHub via the API.

## 1. Upload the Excel workbook

In [ ]:
from google.colab import files

uploaded = files.upload()
xlsx_name = next(iter(uploaded))
print(f"Uploaded: {xlsx_name}  ({len(uploaded[xlsx_name]):,} bytes)")

## 2. Build `data.pkl`

In [ ]:
import pickle
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

OUTPUT_PKL = "data.pkl"

# Alias map: Current-Portfolio name -> name used in Trades & Performance sheets.
# Add entries when a company rebrands. Both sides matched case-insensitively.
NAME_ALIASES = {
    "beone medicines hk line": "Beigene Ltd",   # rebranded from BeiGene
}

def build_pickle(input_xlsx, output_pkl=OUTPUT_PKL):
    input_xlsx = Path(input_xlsx)
    output_pkl = Path(output_pkl)

    trades = pd.read_excel(input_xlsx, sheet_name="Trades")
    perf   = pd.read_excel(input_xlsx, sheet_name="Performance")

    try:
        current_pf_raw = pd.read_excel(input_xlsx, sheet_name="Current Portfolio")
        current_raw = current_pf_raw["Instrument Name"].dropna().unique().tolist()
    except (ValueError, KeyError):
        current_raw = []
        print("⚠ No 'Current Portfolio' sheet found - the toggle will be hidden.")

    trades["Earliest Trade Date"] = pd.to_datetime(trades["Earliest Trade Date"])
    trades = trades.dropna(subset=["Earliest Trade Date", "Instrument Name"]).reset_index(drop=True)

    friendly = perf["Instrument Name"].dropna().tolist()
    price_cols = perf.columns[3:].tolist()
    if len(friendly) != len(price_cols):
        raise ValueError(
            f"Mapping length mismatch: {len(friendly)} friendly names vs "
            f"{len(price_cols)} price columns. Check the workbook layout."
        )
    name_to_col = dict(zip(friendly, price_cols))

    raw = perf[["Name"] + price_cols].rename(columns={"Name": "Date"})
    raw["Date"] = pd.to_datetime(raw["Date"])
    raw = raw.dropna(subset=["Date"]).set_index("Date").sort_index()

    col_to_name = {v: k for k, v in name_to_col.items()}
    prices = raw.rename(columns=col_to_name)

    proxy_target = "Alibaba Group Holding Sponsored ADR"
    if proxy_target in prices.columns:
        prices["Alibaba (HK Line)"] = prices[proxy_target]

    prices = prices.apply(pd.to_numeric, errors="coerce")

    trade_companies = set(trades["Instrument Name"].unique())
    missing = sorted(trade_companies - set(prices.columns))
    if missing:
        print(f"⚠ {len(missing)} trade companies have no price column: {missing}")
    else:
        print("✓ Every trade has a matching price column.")

    # Match current portfolio names to trade names (alias map first, then case-insensitive)
    trade_names_lower = {n.lower(): n for n in trade_companies}
    current_portfolio_names = set()
    unmatched_current = []
    for n in current_raw:
        key = n.lower()
        aliased = NAME_ALIASES.get(key)
        canonical = trade_names_lower.get(aliased.lower()) if aliased else trade_names_lower.get(key)
        if canonical is not None:
            current_portfolio_names.add(canonical)
        else:
            unmatched_current.append(n)

    if current_raw:
        print(f"✓ Current portfolio: {len(current_portfolio_names)}/{len(current_raw)} holdings have trade history.")
        if unmatched_current:
            print(f"  ({len(unmatched_current)} held but never traded: {unmatched_current})")

    payload = {
        "trades": trades,
        "prices": prices,
        "alibaba_hk_proxy": proxy_target,
        "current_portfolio_names": sorted(current_portfolio_names),
        "current_portfolio_raw": sorted(current_raw),
        "generated_at_utc": datetime.now(timezone.utc),
        "source_filename": input_xlsx.name,
        "n_trades": int(len(trades)),
        "n_companies": int(prices.shape[1]),
        "n_price_dates": int(len(prices)),
        "n_current_holdings": int(len(current_raw)),
        "trade_date_range": (
            trades["Earliest Trade Date"].min().to_pydatetime(),
            trades["Earliest Trade Date"].max().to_pydatetime(),
        ),
        "price_date_range": (
            prices.index.min().to_pydatetime(),
            prices.index.max().to_pydatetime(),
        ),
        "schema_version": 2,
    }

    with open(output_pkl, "wb") as f:
        pickle.dump(payload, f, protocol=4)

    size_kb = output_pkl.stat().st_size / 1024
    print(f"\n✓ Wrote {output_pkl}  ({size_kb:,.0f} KB)")
    print(f"   Trades             : {payload['n_trades']:>6,}")
    print(f"   Companies          : {payload['n_companies']:>6,}")
    print(f"   Current holdings   : {payload['n_current_holdings']:>6,}  ({len(current_portfolio_names)} matched to trades)")
    print(f"   Price-data rows    : {payload['n_price_dates']:>6,}")
    print(f"   Trade date range   : {payload['trade_date_range'][0].date()} → {payload['trade_date_range'][1].date()}")
    print(f"   Price date range   : {payload['price_date_range'][0].date()} → {payload['price_date_range'][1].date()}")
    print(f"   Generated (UTC)    : {payload['generated_at_utc'].strftime('%Y-%m-%d %H:%M:%S')}")
    return payload

payload = build_pickle(xlsx_name, OUTPUT_PKL)

## 3. Download `data.pkl`

In [ ]:
files.download(OUTPUT_PKL)

---
## 4. (Optional) Push directly to GitHub

If you'd rather not touch git locally, this cell uploads `data.pkl` straight to your repo via the GitHub API.

**One-time setup:**
1. Generate a fine-grained personal access token at <https://github.com/settings/tokens?type=beta> with **Read & write** permission on **Contents** for the target repo.
2. In Colab's left sidebar, open the **🔑 Secrets** panel and add a secret named `GITHUB_TOKEN` with that value. Toggle "Notebook access" on.
3. Edit `REPO`, `BRANCH`, `PATH_IN_REPO` below to match your setup.

In [ ]:
import base64, json, requests
from google.colab import userdata

REPO         = "your-username/your-repo"
BRANCH       = "main"
PATH_IN_REPO = "data.pkl"
COMMIT_MSG   = f"Refresh data.pkl ({payload['price_date_range'][1].date()})"

token = userdata.get("GITHUB_TOKEN")
if not token:
    raise RuntimeError("Add a Colab secret named GITHUB_TOKEN first.")

api = f"https://api.github.com/repos/{REPO}/contents/{PATH_IN_REPO}"
hdr = {"Authorization": f"Bearer {token}",
       "Accept": "application/vnd.github+json",
       "X-GitHub-Api-Version": "2022-11-28"}

sha = None
r = requests.get(api, headers=hdr, params={"ref": BRANCH})
if r.status_code == 200:
    sha = r.json()["sha"]
elif r.status_code != 404:
    r.raise_for_status()

with open(OUTPUT_PKL, "rb") as f:
    content_b64 = base64.b64encode(f.read()).decode()

body = {"message": COMMIT_MSG, "content": content_b64, "branch": BRANCH}
if sha:
    body["sha"] = sha

r = requests.put(api, headers=hdr, data=json.dumps(body))
r.raise_for_status()
print(f"✓ Pushed to {REPO}@{BRANCH}:{PATH_IN_REPO}")
print(f"  Commit: {r.json()['commit']['sha'][:7]}  ({COMMIT_MSG})")